# Fitandsleek LoRA Training (Colab)

1. **Runtime → Change runtime type → GPU (T4) → Save**
2. **Runtime → Restart session** (important after errors)
3. **Runtime → Run all**
4. Download `fitandsleek-lora.zip` when finished

Repo: https://github.com/kalapak-team/fitandsleek_training_colab

## 1) Check GPU

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU. Go to Runtime → Change runtime type → GPU (T4)')

## 2) Clone / update GitHub repo

In [ ]:
import os
REPO = 'fitandsleek_training_colab'
URL = 'https://github.com/kalapak-team/fitandsleek_training_colab.git'

if not os.path.isdir(REPO):
    !git clone {URL}
%cd {REPO}
!git fetch origin
!git reset --hard origin/main
!grep -n "TRAIN_FIX_V2" scripts/train_lora.py
!ls data scripts

## 3) Install training packages

In [ ]:
!pip install -q -U "transformers>=4.51.0" datasets peft accelerate bitsandbytes trl

## 4) Refresh dataset

In [ ]:
!python scripts/prepare_train_data.py
!wc -l data/fitandsleek_train.jsonl

## 5) Train LoRA
Takes ~15–40 minutes on T4.

In [ ]:
!python scripts/train_lora.py
!ls -la models/fitandsleek-lora

## 6) Zip adapter for download

In [ ]:
import os
from google.colab import files

adapter = 'models/fitandsleek-lora'
marker = os.path.join(adapter, 'adapter_config.json')
if not os.path.isfile(marker):
    raise SystemExit('Training failed: adapter_config.json not found. Re-run Train cell.')

!rm -f fitandsleek-lora.zip
!zip -r fitandsleek-lora.zip models/fitandsleek-lora
files.download('fitandsleek-lora.zip')
print('Download started: fitandsleek-lora.zip')
print('On PC: unzip into models/fitandsleek-lora then set LORA_ADAPTER=models/fitandsleek-lora in .env')